# 02 Extract and clean TAD boundaries

Annotate TAD boundaries with hierarchy levels, remove any boundaries overlapping bad bins.

In [1]:
import pandas as pd
import numpy as np
import bioframe as bf
import cooler
import cooltools
from functools import reduce
from natsort import natsort_keygen

## Extract boundaries

### Read TADs annotation

In [2]:
def read_ontad_annot(fname):
    df = bf.read_table(fname, schema='bedpe')\
           .rename(columns={'name': 'TADlevel', 'score': 'TADmean', 'strand1': 'TADscore'})\
           .drop('strand2', axis=1)
    return df

In [3]:
tads = read_ontad_annot('nested_tads.bedpe')

In [4]:
tads.head()

,chrom1,start1,end1,chrom2,start2,end2,TADlevel,TADmean,TADscore
0,chr2,25000,675000,chr2,25000,675000,1,1.995,2.571
1,chr2,25000,525000,chr2,25000,525000,2,1.470,1.407
2,chr2,25000,165000,chr2,25000,165000,3,2.555,0.511
3,chr2,165000,275000,chr2,165000,275000,3,1.625,0.345
4,chr2,275000,525000,chr2,275000,525000,3,2.065,0.468


### Extract boundaries from TADs

In [5]:
def extract_ontad_boundaries(df, binsize):
    sides = list()
    for coord in ('start1', 'end1'):
        side = df[['chrom1', coord, 'TADlevel']]\
                 .assign(start=lambda df: df[coord] - (binsize // 2),
                         end=lambda df: df[coord] + (binsize // 2))\
                 .rename(columns={'chrom1': 'chrom'})\
                 .drop(coord, axis=1)
        sides.append(side)

    merged = pd.concat(sides, ignore_index=True)
    # we assign boundary level as the smallest level number (i.e. the highest level) of TADs it comprises
    agg = merged.groupby(['chrom', 'start', 'end'])['TADlevel'].min().reset_index()
    return agg

In [6]:
boundaries = extract_ontad_boundaries(tads, 10_000)

In [37]:
boundaries.shape[0]

14107

In [7]:
boundaries.head()

,chrom,start,end,TADlevel
0,chr1,710000,720000,1
1,chr1,840000,850000,1
2,chr1,920000,930000,1
3,chr1,1030000,1040000,2
4,chr1,1210000,1220000,1


### Assign level groups to boundaries

High: 1, Med: 2-3, Low: 4+.

In [8]:
boundaries['levelGroup'] = boundaries['TADlevel'].map(lambda x: 'high' if x == 1 else 'med' if x in (2, 3) else 'low')\
                                                 .astype(pd.CategoricalDtype(['high', 'med', 'low'], ordered=True))

In [9]:
boundaries.shape

(14107, 5)

## Clean boundaries

### Remove boundaries overlapping bad bins

In [32]:
clr_wt = cooler.Cooler('/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2/WT_G2.all.mcool::/resolutions/10000')

In [33]:
bad_bins = clr_wt.bins()[:].query('weight.isna()').reset_index(drop=True)

Remove boundaries that are right next to bad bins as well b/c OnTAD assigns bad bins as TAD boundaries sometimes.

In [34]:
clean_boundaries = bf.closest(boundaries, bad_bins)\
                     .query('distance > 0')\
                     [boundaries.columns]\
                     .reset_index(drop=True)\
                     .drop_duplicates()

In [35]:
clean_boundaries.shape[0]

13545

We removed ca. 500 boundaries.

### Remove boundaries that don't have enough valid pixels in diamonds centered at them

Mimicking a filtering procedure for boundaries from cooltools. We'll use insulation module to calculate the number of valid pixels in diamonds of 3 sizes (50Kb, 100Kb, 250Kb) and choose only those boundaries that have at least 2/3 of valid pixels in each diamond.

In [14]:
hg19_centromeres = bf.fetch_centromeres('hg19')
hg19_chromsizes = clr_wt.chromsizes
hg19_chromarms = bf.make_chromarms(hg19_chromsizes, hg19_centromeres)

In [15]:
insulation_table = cooltools.insulation(clr_wt, [50_000, 100_000, 250_000], view_df=hg19_chromarms, nproc=8)

INFO:root:creating a Pool of 8 workers


In [16]:
valid_pixels_table = insulation_table[['chrom', 'start', 'end'] + [f"n_valid_pixels_{w}" for w in (50_000, 100_000, 250_000)]].copy()

factor_thresh = 2 / 3

print('Size', 'N max', 'N thresh', sep='\t')
for w in (50_000, 100_000, 250_000):
    col_name = f"n_valid_pixels_{w}"
    max_valid_px = valid_pixels_table[col_name].max()
    count_thresh = max_valid_px * factor_thresh
    print(f"{w // 1000} Kb", "%d" % max_valid_px, "%.3f" % count_thresh, sep='\t')
    valid_pixels_table[f"is_valid_bin_{w}"] = valid_pixels_table[col_name] >= count_thresh

Size	N max	N thresh
50 Kb	22	14.667
100 Kb	97	64.667
250 Kb	622	414.667


In [17]:
boundaries_w_valid_px = pd.merge(clean_boundaries,
                                 valid_pixels_table.drop([f"n_valid_pixels_{w}" for w in (50_000, 100_000, 250_000)], axis=1),
                                 on=('chrom', 'start', 'end'),
                                 how='left')
boundaries_w_valid_px[[f"is_valid_bin_{w}" for w in (50_000, 100_000, 250_000)]].value_counts(sort=False).sort_index().rename('count').reset_index()
valid_bin_mask = reduce(np.logical_and, [boundaries_w_valid_px[f"is_valid_bin_{w}"]for w in (50_000, 100_000, 250_000)])

In [18]:
cleaner_boundaries = boundaries_w_valid_px.loc[valid_bin_mask, boundaries.columns].reset_index(drop=True)

In [19]:
cleaner_boundaries.shape[0]

13259

### Remove chrY

HeLa cells are XX, so anything aligning to chrY is an artifact. We should discard these regions.

In [20]:
cleaner_boundaries['chrom'].value_counts().sort_index()

chr1     1073
chr10     606
chr11     603
chr12     658
chr13     446
chr14     430
chr15     396
chr16     336
chr17     416
chr18     357
chr19     329
chr2     1102
chr20     298
chr21     158
chr22     177
chr3      911
chr4      908
chr5      780
chr6      815
chr7      707
chr8      630
chr9      549
chrX      572
chrY        2
Name: chrom, dtype: int64

Indeed, we have only two regions on chrY.

In [21]:
cleanest_boundaries = cleaner_boundaries.query('chrom != "chrY"').reset_index(drop=True)

## Save boundaries

In [22]:
cleanest_boundaries['levelGroup'].value_counts(sort=False)

high    3028
med     5900
low     4329
Name: levelGroup, dtype: int64

In [23]:
cleanest_boundaries = cleanest_boundaries.sort_values(['chrom', 'start'], key=natsort_keygen()).reset_index(drop=True)

In [24]:
cleanest_boundaries.to_parquet('boundaries.parquet')

## REMOVE BEFORE PUB: Compare with previous boundaries

In [25]:
prev_boundaries = pd.read_parquet('/groups/gerlich/experiments/Experiments_006300/006326/data/output/50_get_and_clean_nested_boundaries_1e-1/ontad_boundaries.penalty_1e-1.all.cleaner.parquet')

In [26]:
merged = pd.merge(cleanest_boundaries, prev_boundaries.query('chrom != "chrY"'), on=('chrom', 'start', 'end'), how='outer')

In [27]:
merged.head()

,chrom,start,end,TADlevel_x,levelGroup,TADlevel_y,TADlevelCode
0,chr1,4570000,4580000,7.0,low,7.0,>=6
1,chr1,4700000,4710000,7.0,low,7.0,>=6
2,chr1,5570000,5580000,5.0,low,5.0,5
3,chr1,5690000,5700000,3.0,med,3.0,3
4,chr1,5920000,5930000,2.0,med,2.0,2


In [28]:
(merged['TADlevel_x'] != merged['TADlevel_y']).mean()

0.011263613307474265

1% difference! How come?

In [31]:
merged.query('TADlevel_x != TADlevel_y')

,chrom,start,end,TADlevel_x,levelGroup,TADlevel_y,TADlevelCode
11449,chr18,14530000,14540000,2.0,med,NaN,NaN
13173,chrX,129660000,129670000,4.0,low,NaN,NaN
13257,chr1,920000,930000,NaN,NaN,1.0,1
13258,chr1,1030000,1040000,NaN,NaN,2.0,2
13259,chr1,1290000,1300000,NaN,NaN,1.0,1
...,...,...,...,...,...,...,...
13401,chrX,140370000,140380000,NaN,NaN,3.0,3
13402,chrX,140500000,140510000,NaN,NaN,2.0,2
13403,chrX,140780000,140790000,NaN,NaN,4.0,4
13404,chrX,143150000,143160000,NaN,NaN,2.0,2


The reason is that for the previous dataset I used WT G1 cooler to remove bad bins and invalid diamonds. Remarkably similar annotation!